# 02 · Standard FunnyBird CBM: controlled discovery of concept backwash

**Report question.** When one FunnyBird part is replaced while body, pose,
camera, and background stay fixed, does the corresponding concept answer
follow the inserted part or remain attached to the old bird?

**Population.** Standard non-RL CBM. This notebook contains no MCBM and no
visibility-aware relabelled model. Seed-level replication is shown where
accepted outputs exist; the fixed-render causal analysis begins with seed 1.

**Claims available here.** FunnyBird's renderer permits a controlled
donor-part replacement. Therefore a validated positive donor response plus
a remaining source preference can establish the CBM backwash event. Proposed
explanations are weaker unless independently manipulated.


## What this notebook must prove, and how the figures build the claim

The final claim is deliberately narrower than “the model is inaccurate.” A
FunnyBird **backwash event** requires both of the following on the *same
controlled replacement*:

1. the donor part moves the raw concept comparison toward the donor
   (`response_delta > 0`); and
2. after that movement, the old source concept is still higher
   (`m_cf < 0`).

Example: replacing a red tail with a blue tail raises the blue-tail score
relative to red by 24 units, but red still finishes 6 units above blue. The
model reacted to the new tail pixels, yet its final concept answer remained
attached to the old bird. That is the event tested in Figure 4.

Before accepting it, the report must establish these components in order:

| Step | Needed fact | Figure(s) | Why it is needed |
|---|---|---|---|
| 1 | the trained concept outputs are usable | 1 | a constant or broken output cannot support grounding analysis |
| 2 | the renderer really changed only the named part | 2 | otherwise a score change cannot be assigned to that part |
| 3 | the inserted pixels cause donorward movement | 3 | proves the model saw some evidence in the new part |
| 4 | the old source can still win after that movement | 4 | this is the controlled backwash predicate |
| 5 | the event is not a direction-averaging artifact | 5 | checks forward and reverse replacements separately |
| 6 | test proposed contributors | 6–8 | visibility/occlusion, conflicting labels, exact-value difficulty, support/alternatives, and source species |
| 7 | measure what those contributors predict and what remains | 9 | prevents claiming that a plausible story explains all rows |
| 8 | measure downstream class impact | 10 | separates explanation failure from species-classification harm |

### The three contributor hypotheses carried into both reports

The linked comparison tests the same three proposed reasons in the same order:

1. **visibility/occlusion:** the named pixels may be absent or too small;
2. **label–visibility conflict:** training may call a concept positive when its
   mapped region is not visible; and
3. **exact-value difficulty:** some variants may be intrinsically harder, rarer,
   or drawn from a larger alternative set.

Only after those are measured do we ask whether unchanged source species/body
context organizes the remaining raw-score error.  That fourth term is a
residual association, not a promise that the three measured reasons sum to the
whole phenomenon.

The implementation retains the complete renderer audit, all exact values,
species residuals, recall/model-health controls, and provenance inherited from
the earlier curated report and the original renderer-swap and recall notebooks.

### Capabilities and limits that determine this design

FunnyBird supplies an exact renderer mask and a clean donor-part replacement:
body, pose, camera, and background can remain unchanged while one part changes.
That makes Figures 3–4 causal tests of the changed part pixels. Visibility,
training-label conflict, exact value, support, and species are then investigated
as possible contributors. Except for the later matched RLv2 retraining, those
contributor analyses are observational and are not allowed to erase the
controlled event or claim that every cause has been found.

### Predictions stated before the results

- If the concept is locally grounded, replacement should produce
  `response_delta > 0` and usually `m_cf > 0`.
- If backwash occurs, a nontrivial set should have `response_delta > 0` but
  `m_cf < 0`.
- If visibility/occlusion is sufficient, the event should disappear for large,
  clearly visible inserted parts.
- If label–visibility conflict contributes, parts with more positive labels on
  invisible parts should later improve most under matched RLv2 training.
- If exact-value difficulty or species context contributes, matched rows should
  retain systematic value- or species-linked differences.
- None of these predictions requires the measured contributors to reduce the
  remaining error to zero.


## The implemented CBM and the notation used below

For image `i`, the encoder produces a latent concept vector `h_i` with one
slot for each of the `J` exact concepts.  The learned concept head for slot `j`
turns `h_ij` into the raw concept logit `z_ij`:

```text
x_i → image encoder → h_i = (h_i1, …, h_iJ)
                          ├→ learned head q_j(h_ij) → z_ij → sigmoid → p_ij
                          └→ class head on complete h_i       → species prediction
```

The implementation trains with

`L_CBM = L_task + beta × L_concept`.

The class head reads the complete latent vector `h_i`; it does not read a list of
hard 0/1 concept decisions. Thus species loss can shape the same latent slots
that the concept heads read. In these runs each concept head is a learned
`1 → 3 → 1` network, not the identity. The setup cell replays the saved head
weights on saved `h_i` and verifies that `sigmoid(z_ij)` exactly reproduces
the saved probability.

| Symbol | Meaning |
|---|---|
| `x_i` | image `i` |
| `y_i` | species label |
| `c_ij` | processed 0/1 label for exact concept `j` |
| `h_ij` | encoder's latent slot for concept `j`; also read by the class head |
| `z_ij = q_j(h_ij)` | raw concept logit after the learned head; primary grounding quantity |
| `p_ij = sigmoid(z_ij)` | bounded probability; used only for thresholded performance |
| `c_hat_ij = 1[z_ij>0]` | predicted concept presence |
| `v_ig` | whether mapped part mask `g` is visible |
| `a_ig` | visible area of mask `g` |

`L_task` is the species-classification loss. `L_concept` is the sum of the
per-concept label losses. `beta` controls their relative weight. No later plot
uses the encoder slot `h_ij` while calling it a concept logit: grounding plots
use the post-head raw score `z_ij`.

Ordinary accuracy and recall answer whether predictions agree with labels. They
do **not** answer whether the prediction came from the named pixels.


## Dataset design and report population

FunnyBird is synthetic, so the relevant objects are known exactly rather than
estimated from photographs.

| Item | Value used here | Why it matters |
|---|---:|---|
| species | 50 | unchanged species/body appearance is the possible contextual signal |
| named parts | `tail`, `wing`, `beak`, `foot`, `eye` | these are the only five FunnyBird part names used below |
| exact concepts | 26 part values across the five parts | for example, `tail::blue`; a part and its exact value are not interchangeable |
| held-out model-health population | 5,000 test images | used for Figure 1 and the species decoder |
| controlled swap population | accepted fixed-render seed-1 CSV | the same validated rendered images are reused across model comparisons |

Species determine part values in FunnyBird, so species context can predict a
concept label even when the named part is hard to see. That makes contextual
prediction possible, but it does not prove the trained CBM used context. The
controlled replacement in Figures 2–4 supplies that stronger test.


In [ ]:
import os, json, re, glob, sys, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as DisplayImage

CURATED = Path(os.environ["CURATED_DATA"])
CWD = Path.cwd()
REPO = CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0, str(REPO/"data"/"funnybirds"))
plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 40)
ORDER = ["tail", "wing", "beak", "foot", "eye"]
COLORS = {"tail":"#6A0DAD", "wing":"#0072B2", "beak":"#E69F00",
          "foot":"#009E73", "eye":"#CC79A7"}

def require(path, command):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}\nProduce it with: {command}")
    return path

swap_candidates = [
    CURATED/"swap_fixed_v3_matched"/"funnybirds-cbm-s1.csv",
    CURATED/"swap_fixed_v2_attempt2"/"funnybirds-cbm-s1.csv",
    CURATED/"swap_fixed_v2"/"funnybirds-cbm-s1.csv",
]
SWAP = next((p for p in swap_candidates if p.exists()), None)
if SWAP is None:
    raise FileNotFoundError("No accepted fixed-render standard-CBM swap CSV")
S = pd.read_csv(SWAP)
# Legacy column names start with z_, but the renderer driver stored
# model output["c_logits"] after the learned concept head.  In report
# notation these are z_source and z_donor, not latent h values.
if "response_delta" not in S:
    S["response_delta"] = S.margin - (S.z_new_orig - S.z_old_orig)
S["responded_but_source_wins"] = (S.response_delta > 0) & (S.margin < 0)
print("fixed-render input:", SWAP)
print("rows:", len(S), "parts:", sorted(S.part.unique()))

PRED_DIR = REPO/"external"/"minimal_cbm"/"results"/"funnybirds-cbm"/"1"/"predictions"
PRED = require(PRED_DIR/"epoch_100.pth", "train or restore funnybirds-cbm seed 1 epoch 100")
import torch
saved = torch.load(PRED, map_location="cpu", weights_only=False)
latent_h_saved = saved["z"].detach().cpu().numpy().reshape(len(saved["z"]), -1)
p_saved = saved["c_preds"].detach().cpu().numpy().reshape(len(saved["c_preds"]), -1)
c_saved = saved["c"].detach().cpu().numpy().reshape(len(saved["c"]), -1)
MODEL = require(REPO/"external"/"minimal_cbm"/"results"/"funnybirds-cbm"/"1"/"models"/"epoch_100.pt",
                "train or restore funnybirds-cbm seed 1 epoch 100")
sys.path.insert(0, str(REPO/"analysis"))
from minimal_cbm_scores import concept_logits_from_saved_latent, validate_saved_probabilities
logit_tensor = concept_logits_from_saved_latent(saved["z"], MODEL, c_saved.shape[1])
head_error = validate_saved_probabilities(logit_tensor, saved["c_preds"])
z_saved = logit_tensor.numpy()
print(f"[CONCEPT-HEAD REPLAY PASS] max |sigmoid(raw_logit)-saved_prob|={head_error:.3g}")

FB_ROOT = Path(os.environ.get("FUNNYBIRDS_ROOT", CURATED/"FunnyBirds"))
import funnybirds_concepts as fbc
parts = fbc.load_parts(FB_ROOT)
CONCEPT_NAMES = fbc.concept_names(parts)
SPANS = fbc.group_slices(parts)
if len(CONCEPT_NAMES) != z_saved.shape[1]:
    raise RuntimeError("parts.json concept width does not match saved predictions")
CONCEPT_PART = {name: part for part,(a,b) in SPANS.items() for name in CONCEPT_NAMES[a:b]}
print("checkpoint:", PRED, "concepts:", len(CONCEPT_NAMES), "species:", len(np.unique(saved["y"])))


## 1 · Did training produce a usable, non-collapsed CBM?

**Question.** Did training produce a usable, non-collapsed CBM?

**Variables and prediction.** For every exact concept `j`, measure raw-score spread, positive-versus-negative label separation, balanced accuracy, and positive recall. A usable slot has nonzero spread, positive label separation, and above-chance thresholded performance.

**Method.** Compute all quantities from the epoch-100 held-out predictions. Recall is a health statistic, not grounding evidence.

### Figure 1 · Did training produce a usable, non-collapsed CBM?

**How to read the figure.** Each row is one exact concept, such as `yellow tail`. The four panels use the
same rows. `spread = Q95(z)-Q05(z)` asks whether the output changes across
test images; exactly zero means a constant output. `label separation =
median(z|c=1)-median(z|c=0)` asks how far positive-labelled images sit above
negative-labelled images; positive is the expected direction. `balanced
accuracy = (positive recall + negative recall)/2` gives positive and negative
labels equal weight; 0.5 is chance for a binary concept. `positive recall =
P(z>0|c=1)` is the fraction of labelled-positive images called positive.
Example: positive recall 0.90 means 90 of 100 positive-labelled images have
`z>0`. Dot color identifies the FunnyBird part: purple tail, blue wing,
orange beak, green foot, and pink eye. The solid zero line marks no label
separation; the dashed 0.5 lines mark chance balanced accuracy and 50%
positive recall. These are health checks, not evidence about which pixels
produced `z`.


In [ ]:
# ALT: Four aligned dot plots showing raw-score spread, label separation, balanced accuracy, and positive recall for every FunnyBird concept.
def balanced_accuracy(y, pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

rows=[]
for j,name in enumerate(CONCEPT_NAMES):
    z=z_saved[:,j]; c=c_saved[:,j].astype(int); pred=(z>0).astype(int)
    rows.append({"concept":name,"part":CONCEPT_PART[name],
                 "spread":np.quantile(z,.95)-np.quantile(z,.05),
                 "label_separation":np.median(z[c==1])-np.median(z[c==0]),
                 "balanced_accuracy":balanced_accuracy(c,pred),
                 "positive_recall":pred[c==1].mean(),
                 "n_positive":int(c.sum()),"n_negative":int((c==0).sum())})
HEALTH=pd.DataFrame(rows).sort_values(["part","concept"])
y_true=np.asarray(saved["y"]).reshape(-1).astype(int)
y_scores=np.asarray(saved["y_preds"])
if y_scores.ndim>2: y_scores=y_scores.reshape(len(y_scores),-1)
task_accuracy=float((y_scores.argmax(1)==y_true).mean())
concept_accuracy=float(((z_saved>0)==c_saved).mean())
display(pd.DataFrame([{"images":len(y_true),"species":len(np.unique(y_true)),
                      "task_accuracy":task_accuracy,"concept_accuracy":concept_accuracy}]).round(4))
display(HEALTH.round(3))
metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(15,max(5,.24*len(HEALTH))),sharey=True)
y=np.arange(len(HEALTH))
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=HEALTH.part.map(COLORS).fillna("#BBBBBB"),s=24)
    ax.set_xlabel(m.replace("_"," "))
    if m in ["label_separation"]: ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept,fontsize=7)
axes[0].invert_yaxis(); fig.suptitle("Figure 1 · Exact-concept model-health guard")
plt.tight_layout(); plt.show()


### Review record for Figure 1

- **Literal observation:** All 26 exact outputs vary substantially: raw-z spread is 20.77-56.88, label separation is 20.71-37.68, balanced accuracy is 0.966-1.000, and positive recall is 0.950-1.000. Image classification accuracy is 0.7504 and concept accuracy is 0.9944.
- **Strongest alternative explanation:** Excellent label prediction can still come from species context rather than the named part, so this figure establishes health but not grounding.
- **Discriminating test:** Use the same-image controlled replacement in Figures 3-4.
- **Limited conclusion:** `ACCEPTED FOR seed-1 standard-CBM model health; no exact output is collapsed.`
- **Next question:** Is the fixed renderer intervention itself valid?


## 2 · Did the renderer change only the intended part?

**Question.** Did the renderer change only the intended part?

**Variables and prediction.** Inspect the semantic preflight and original/swap/delete/part-map examples for all five parts. A valid intervention visibly changes the target part, preserves the rest of the scene, and has nonzero target-mask pixels.

**Method.** Use artifacts from the accepted fixed-render root before reading any model response.

### Figure 2 · Did the renderer change only the intended part?

**How to read the figure.** Figure 2a is the semantic preflight: for each of the five parts it shows the
original, replacement, deletion, original part map, and replacement part
map. Figure 2b is the compact four-column audit used by the analysis:
original, replacement, deletion, and replacement part map. In both grids the
named part should change while body, pose, camera, and background remain
fixed. This validates the intervention before model scores are interpreted.


In [ ]:
# ALT: Complete FunnyBird intervention audit showing original, swapped, deleted, and part-map images for tail, wing, beak, foot, and eye.
ROOT = SWAP.parent
preflight_candidates=[ROOT/"renderer_preflight"/"renderer_semantic_preflight.png",
                      CURATED/"swap_fixed_v2_attempt2"/"renderer_preflight"/"renderer_semantic_preflight.png"]
preflight=next((p for p in preflight_candidates if p.exists()),preflight_candidates[0])
example_candidates=[ROOT/"examples",CURATED/"swap_fixed_v2_attempt2"/"examples"]
examples=next((p for p in example_candidates if p.is_dir()),example_candidates[0])
from PIL import Image
if preflight.exists():
    im0=Image.open(preflight).convert("RGB")
    fig0,ax0=plt.subplots(figsize=(14,3.2))
    ax0.imshow(im0); ax0.axis("off")
    ax0.set_title("Figure 2a · Semantic preflight: original, swap, delete, original map, swap map")
    plt.tight_layout(); plt.show()
else:
    print("preflight sheet not stored beside CSV; use accepted job-3330289 audit")
tags=["orig","swap","delete","swap_partmap"]
fig,axes=plt.subplots(len(ORDER),len(tags),figsize=(12,13))
for r,part in enumerate(ORDER):
    for c,tag in enumerate(tags):
        ax=axes[r,c]; files=sorted(examples.glob(f"{part}_*_{tag}.png"))
        if files: ax.imshow(Image.open(files[0]).convert("RGB"))
        else: ax.text(.5,.5,"missing",ha="center",va="center")
        ax.set_title(f"{part} · {tag}"); ax.axis("off")
fig.suptitle("Figure 2b · Complete intervention audit: original, replacement, deletion, and target mask")
plt.tight_layout(); plt.show()


### Review record for Figure 2

- **Literal observation:** For tail, wing, beak, foot, and eye, the displayed replacement and deletion alter the named part while the body, pose, camera, and background remain fixed; the target part map contains the changed region.
- **Strongest alternative explanation:** A few photographs alone would not certify the full cache.
- **Discriminating test:** Retain the semantic preflight plus the accepted fixed-render hash/diversity validation across all evaluated models and render IDs.
- **Limited conclusion:** `ACCEPTED FOR the validated FunnyBird fixed-render intervention.`
- **Next question:** Do those inserted pixels move the raw concept comparison toward the donor?


## 3 · Did the inserted pixels move the comparison toward the donor?

**Question.** Did the inserted pixels move the comparison toward the donor?

**Variables and prediction.** `response_delta = (z_donor-z_source)_cf - (z_donor-z_source)_orig`. Legacy CSV columns named `z_*` contain these post-head raw logits. Values above zero mean that replacement pixels moved the model toward the donor concept.

**Method.** Plot the complete distribution for every part and report the positive-response rate.

### Figure 3 · Did the inserted pixels move the comparison toward the donor?

**How to read the figure.** Panel A puts part on the x-axis and `response_delta` in raw-logit units on the
y-axis. The box spans the 25th--75th percentiles, the orange line is the
median, and whiskers are the 5th--95th percentiles; outliers are omitted only
from drawing. Zero means no donorward change and values above zero mean the
donor gained relative to the old source. Panel B reports the fraction above
zero, with `n` printed over each bar. Colors identify parts using the shared
FunnyBird palette. This measures response size, not whether the donor wins.
Example: a margin change from -20 before replacement to -5 afterward gives
`response_delta=+15`, although the final margin remains negative.


In [ ]:
# ALT: FunnyBird response-delta distributions and positive donor-response rates for all five parts.
fig,axes=plt.subplots(1,2,figsize=(12,4.2))
vals=[S.loc[S.part==p,"response_delta"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("response_delta (raw logit units)")
axes[0].set_title("A · Distribution of donorward movement")
rate=S.groupby("part").response_delta.apply(lambda x:(x>0).mean()).reindex(ORDER)
axes[1].bar(rate.index,rate.values,color=[COLORS[p] for p in rate.index])
axes[1].set_ylim(0,1)
axes[1].set_ylabel("fraction with response_delta > 0"); axes[1].set_title("B · Positive donor-response rate")
counts=S.groupby("part").size().reindex(ORDER)
for x,(p,v) in enumerate(rate.items()): axes[1].text(x,v+.025,f"n={counts[p]:,}",ha="center",fontsize=8)
fig.suptitle("Figure 3 · Does the replacement produce the predicted within-image response?")
plt.tight_layout(); plt.show(); display(rate.rename("positive_response_rate").to_frame().round(3))


### Review record for Figure 3

- **Literal observation:** Median donorward movement is positive for every part, and the positive-response rates are tail 0.913, wing 0.999, beak 0.997, foot 0.995, and eye 0.993.
- **Strongest alternative explanation:** A positive movement alone does not say that the inserted donor finishes above the old source.
- **Discriminating test:** Inspect the final donor-minus-source margin jointly with response_delta.
- **Limited conclusion:** `ACCEPTED FOR a causal within-image response to the inserted part pixels.`
- **Next question:** After that response, which concept finishes stronger?


## 4 · After responding, does the donor finish above the old source?

**Question.** After responding, does the donor finish above the old source?

**Variables and prediction.** The final margin is `m_cf=z_donor,cf-z_source,cf`. The primary event is `response_delta>0` with `m_cf<0`. A lower-right quadrant point means the inserted pixels had an effect but the old source still wins.

**Method.** Show final-margin distributions and the joint response/margin plane for every part.

### Figure 4 · After responding, does the donor finish above the old source?

**How to read the figure.** In the margin panel, zero separates donor wins (`m_cf>0`) from old-source wins
(`m_cf<0`). In the quadrant panel, x is donorward movement and y is the final
donor-minus-source score. The lower-right quadrant is the controlled
backwash predicate `response_delta>0 and m_cf<0`: the new pixels moved the
answer toward the donor, but the old source still finished higher. Boxes and colors use the Figure 3 definitions;
translucent points are individual swaps and the legend maps color to part.
Example: `m_cf=-5` means the old source finishes five raw-logit units above
the donor.


In [ ]:
# ALT: Final donor-minus-source margin distributions and joint response-delta versus final-margin plot for all FunnyBird parts.
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
vals=[S.loc[S.part==p,"margin"].dropna() for p in ORDER]
bp=axes[0].boxplot(vals,tick_labels=ORDER,showfliers=False,whis=(5,95),patch_artist=True)
for box,p in zip(bp["boxes"],ORDER): box.set_facecolor(COLORS[p]); box.set_alpha(.55)
axes[0].axhline(0,color="black",lw=1); axes[0].set_ylabel("final margin m_cf (donor − source)")
axes[0].set_title("A · Final donor-minus-source margin")
for p in ORDER:
    d=S[S.part==p]
    axes[1].scatter(d.response_delta,d.margin,s=10,alpha=.22,color=COLORS[p],label=p)
axes[1].axvline(0,color="black",lw=1); axes[1].axhline(0,color="black",lw=1)
axes[1].set_xlabel("response_delta"); axes[1].set_ylabel("final margin m_cf")
axes[1].set_title("B · Lower-right = responds, but old source still wins")
axes[1].legend(ncol=5,fontsize=8)
fig.suptitle("Figure 4 · Controlled FunnyBird backwash predicate")
plt.tight_layout(); plt.show()
summary=S.groupby("part").agg(n=("margin","size"),median_response=("response_delta","median"),
    median_final_margin=("margin","median"),positive_response_rate=("response_delta",lambda x:(x>0).mean()),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER)
display(summary.round(3))


### Review record for Figure 4

- **Literal observation:** The median final margin is negative for tail (-7.927) and beak (-1.368), near zero for eye (0.011), and positive for wing (13.665) and foot (13.559). The responded-but-source-wins rates are 0.608, 0.537, 0.491, 0.133, and 0.084, respectively.
- **Strongest alternative explanation:** Starting preference, swap direction, target visibility, exact value difficulty, and source species could organize the unequal rates.
- **Discriminating test:** Test those alternatives separately in Figures 5-9 without changing the event definition.
- **Limited conclusion:** `ACCEPTED FOR the seed-1 controlled FunnyBird backwash predicate, strongest for tail, beak, and eye; wing and foot also contain minority events.`
- **Next question:** Can swap direction create the pooled pattern?


## 5 · Could opposite swap directions create the result?

**Question.** Could opposite swap directions create the result?

**Variables and prediction.** Compare forward and backward rates of `response_delta>0 and final margin<0`, together with median margins. A genuine part pattern should appear in both directions rather than cancel when pooled.

**Method.** Keep directions separate and show their denominators.

### Figure 5 · Could opposite swap directions create the result?

**How to read the figure.** Each part has separate forward (`fwd`) and backward (`bwd`) replacement
estimates, shown as unconnected circles and squares. The rate is
the fraction of rows in the lower-right quadrant from Figure 4; the printed
denominator is the number of swaps. Similar values in both directions argue
against a pooled average hiding opposite effects. A rate of 0.60 means 60%
of swaps in that direction satisfy both `response_delta>0` and `m_cf<0`.


In [ ]:
# ALT: Forward and backward FunnyBird rates where the donor changes the margin but the old source remains larger, alongside final margins for every part.
D=(S.groupby(["part","direction"]).agg(n=("margin","size"),median_margin=("margin","median"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index())
fig,axes=plt.subplots(1,2,figsize=(12,4))
x=np.arange(len(ORDER))
for direction,marker,offset in [("fwd","o",-.10),("bwd","s",.10)]:
    d=D[D.direction==direction].set_index("part").reindex(ORDER)
    axes[0].scatter(x+offset,d.responded_but_source_wins_rate,marker=marker,label=direction,s=45)
    axes[1].scatter(x+offset,d.median_margin,marker=marker,label=direction,s=45)
for ax in axes: ax.set_xticks(x,ORDER)
axes[0].set_ylim(0,1); axes[0].set_ylabel("fraction: donorward response, but source still wins")
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("median final margin")
axes[0].legend(); axes[1].legend(); fig.suptitle("Figure 5 · Forward and backward directions")
plt.tight_layout(); plt.show(); display(D.round(3))


### Review record for Figure 5

- **Literal observation:** Forward and backward results have the same part ordering. Tail remains near 0.60 in both directions, while wing and foot remain near 0.07-0.15; beak and eye remain intermediate.
- **Strongest alternative explanation:** Individual source/donor value pairs can still be asymmetric even when pooled directions agree.
- **Discriminating test:** Inspect every exact inserted value and both direction-specific denominators.
- **Limited conclusion:** `ACCEPTED FOR excluding opposite-direction cancellation as the main explanation.`
- **Next question:** How does exact target visibility change the event?


## 6 · How much of the result is associated with target visibility?

**Question.** How much of the result is associated with target visibility?

**Variables and prediction.** Use `pixel_count_cf` from the exact swapped-part map and the same final-margin and `response_delta>0, margin<0` definition. If visibility is sufficient, highly visible replacements should remove the part gap; a remaining gap requires another explanation.

**Method.** Use declared bins and print the number of swap rows in every bin.

### Figure 6 · How much of the result is associated with target visibility?

**How to read the figure.** The x-axis bins swaps by the number of visible pixels in the inserted target
part. One panel shows median final raw-logit margin; the other shows the
responded-but-source-wins fraction. If visibility were the whole explanation,
sufficiently large visible parts should make margins positive and drive that
fraction near zero for every part. Point color identifies part; the table
gives the exact denominator for every nonempty bin. The companion visible-only
summary uses the same rule for all parts: `pixel_count_cf > 0`. A median
margin of +3 means the donor finishes three raw-logit units above the source.


In [ ]:
# ALT: FunnyBird final margin and responded-but-source-still-wins rate across exact swapped-part visibility bins for all parts.
if "pixel_count_cf" not in S: raise RuntimeError("fixed swap CSV lacks pixel_count_cf")
bins=[0,20,50,100,200,500,np.inf]; labels=["0–19","20–49","50–99","100–199","200–499","500+"]
V=S.copy(); V["visibility_bin"]=pd.cut(V.pixel_count_cf,bins=bins,labels=labels,right=False)
T=V.groupby(["part","visibility_bin"],observed=True).agg(
    n=("margin","size"),median_margin=("margin","median"),responded_but_source_wins_rate=("responded_but_source_wins","mean")).reset_index()
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for p in ORDER:
    d=T[T.part==p].set_index("visibility_bin").reindex(labels)
    axes[0].plot(labels,d.median_margin,"o-",label=p,color=COLORS[p])
    axes[1].plot(labels,d.responded_but_source_wins_rate,"o-",label=p,color=COLORS[p])
axes[0].axhline(0,color="black",lw=.8); axes[0].set_ylabel("median final margin")
axes[1].set_ylim(0,1); axes[1].set_ylabel("fraction: donorward response, but source still wins")
for ax in axes: ax.tick_params(axis="x",rotation=45); ax.legend(fontsize=8,ncol=2)
fig.suptitle("Figure 6 · Same-render visibility analysis")
VISIBLE_ONLY=(V[V.pixel_count_cf>0].groupby("part").agg(
    n_visible_rows=("margin","size"),median_margin=("margin","median"),
    responded_but_source_wins_rate=("responded_but_source_wins","mean")).reindex(ORDER))
plt.tight_layout(); plt.show(); display(T.round(3)); display(VISIBLE_ONLY.round(3))


### Review record for Figure 6

- **Literal observation:** Larger swapped-part masks generally move beak, eye, tail, and wing margins rightward and reduce their source-wins rate. Tail nevertheless remains negative through most bins and is non-monotone in the middle bins; wing and foot are already mostly donor-positive.
- **Strongest alternative explanation:** Pixel count is associated with pose, source/donor value, and species, so bins do not isolate visibility causally by themselves.
- **Discriminating test:** Hold exact values and species fixed, and test the visibility-aware label change later with matched RLv2 training.
- **Limited conclusion:** `ACCEPTED FOR visibility as a contributor, not a sufficient explanation.`
- **Next question:** Did ordinary training assign positive labels when the part was not visible?


## 6b · How often did the original training label conflict with visible part evidence?

**Question.** How often did the original training label conflict with visible part evidence?

**Variables and prediction.** Compare the standard and visibility-aware training records for the same images; count positive concept labels changed to zero within each part group. A large conflict count identifies a plausible training signal that can reward contextual prediction, but its causal effect belongs to notebook 03rl.

**Method.** Require identical ordered image/class records and allow only `attribute_label` to differ.

### Figure 6b · How often did the original training label conflict with visible part evidence?

**How to read the figure.** Each row is one exact concept. The x-axis is
`P(visibility-aware label=0 | original label=1)`: the number of original
positive training labels removed by the visibility rule divided by all
original positive labels for that concept. A value of 0.25 means 25 of 100
positive labels conflict with visible part evidence. Color identifies part.
This is a data rate, not a model probability or causal model effect.


In [ ]:
# ALT: FunnyBird training-image counts whose positive part-concept labels change under the matched visibility-aware relabeling rule.
import pickle
std_path=CURATED/"funnybirds_processed_trainval"/"train.pkl"
rl_path=CURATED/"funnybirds_processed_rl_trainval"/"train.pkl"
if not (std_path.exists() and rl_path.exists()):
    print("INCOMPLETE: matched standard/RLv2 training records are not both present")
else:
    std=pickle.loads(std_path.read_bytes()); rl=pickle.loads(rl_path.read_bytes())
    if len(std)!=len(rl): raise RuntimeError("standard/RLv2 train lengths differ")
    positive=np.zeros(len(CONCEPT_NAMES),dtype=int); changed=np.zeros(len(CONCEPT_NAMES),dtype=int)
    for a,b in zip(std,rl):
        for key in a:
            if key=="attribute_label": continue
            av,bv=a[key],b[key]
            equal=np.array_equal(np.asarray(av),np.asarray(bv)) if isinstance(av,(list,tuple,np.ndarray)) else av==bv
            if not bool(equal): raise RuntimeError(f"non-label record field differs: {key}")
        ca=np.asarray(a["attribute_label"]); cb=np.asarray(b["attribute_label"])
        positive += (ca==1); changed += ((ca==1)&(cb==0))
    CONFLICT_EXACT=pd.DataFrame({"concept":CONCEPT_NAMES,"part":[CONCEPT_PART[n] for n in CONCEPT_NAMES],
        "n_positive":positive,"n_changed":changed})
    CONFLICT_EXACT["conflict_rate"]=CONFLICT_EXACT.n_changed/CONFLICT_EXACT.n_positive.replace(0,np.nan)
    PART_CONFLICT=(CONFLICT_EXACT.groupby("part").agg(n_positive=("n_positive","sum"),
        n_changed=("n_changed","sum")).reindex(ORDER))
    PART_CONFLICT["conflict_rate"]=PART_CONFLICT.n_changed/PART_CONFLICT.n_positive
    q=CONFLICT_EXACT.sort_values(["part","concept"]); y=np.arange(len(q))
    fig,ax=plt.subplots(figsize=(10,max(6,.24*len(q))))
    ax.barh(y,q.conflict_rate,color=q.part.map(COLORS)); ax.set_yticks(y,q.concept,fontsize=7)
    ax.invert_yaxis(); ax.set_xlim(0,1); ax.set_xlabel("positive-label / invisible-mask conflict rate")
    ax.set_title("Figure 6b · Exact-concept training label–visibility conflict")
    plt.tight_layout(); plt.show(); display(q.round(3)); display(PART_CONFLICT.round(3))


### Review record for Figure 6b

- **Literal observation:** Visibility-aware preprocessing removes 6,711 positive tail labels, compared with 334 beak, 247 eye, 42 foot, and 12 wing labels; every change occurs on a distinct training image.
- **Strongest alternative explanation:** These are training-signal counts, not measured causal effects on the trained standard model.
- **Discriminating test:** Compare otherwise matched standard and RLv2 checkpoints on the same fixed renders.
- **Limited conclusion:** `ACCEPTED FOR a measured label/visibility conflict, especially for tail; causal credit remains deferred to notebook 03rl.`
- **Next question:** Are some exact visual variants much harder than others?


## 7 · Do exact source and donor values explain the failures?

**Question.** Do exact source and donor values explain the failures?

**Variables and prediction.** For every part, compare the inserted donor value with the concept value that has the largest post-swap raw score. A clean diagonal means exact visual values are distinguished; recurring bright columns indicate default answers.

**Method.** Display all parts and all values with row-normalized counts.

### Figure 7 · Do exact source and donor values explain the failures?

**How to read the figure.** Each heatmap row is the value actually inserted and each column is the value
with the largest post-swap raw logit. A bright diagonal means the model names
the inserted value; bright off-diagonal cells show systematic confusion.
Every FunnyBird part and every value is included. The lower row gives the
final-margin distribution for the same inserted values, with the number of
swaps printed above each box. Thus recognition and retained-source margin are
visible together rather than inferred from a diagonal rate alone. A diagonal
value of 0.80 means the inserted value is highest in 80% of that row's swaps.


In [ ]:
# ALT: Five row-normalized confusion matrices comparing inserted and highest-scoring FunnyBird part values.
available=[p for p in ORDER if any(c.startswith(f"z_cf_{p}_") for c in S.columns)]
if set(available)!=set(ORDER): raise RuntimeError(f"missing all-part post-swap concept logits: have {available}")
fig,axes=plt.subplots(2,5,figsize=(18,7),constrained_layout=True)
diag={}
value_rows=[]
for col,p in enumerate(ORDER):
    ax=axes[0,col]; bax=axes[1,col]
    cols=sorted([c for c in S if c.startswith(f"z_cf_{p}_")],key=lambda x:int(x.rsplit("_",1)[1]))
    d=S[S.part==p].dropna(subset=cols); donor=d.var_donor.astype(int).to_numpy(); pred=d[cols].to_numpy().argmax(1)
    M=np.zeros((len(cols),len(cols)))
    for a,b in zip(donor,pred):
        if 0<=a<len(cols): M[a,b]+=1
    M=M/np.maximum(M.sum(1,keepdims=True),1); diag[p]=(donor==pred).mean()
    im=ax.imshow(M,vmin=0,vmax=1,cmap="magma"); ax.set_title(f"{p}\ndiagonal={diag[p]:.2f}")
    ax.set_xlabel("highest-scoring value"); ax.set_ylabel("inserted value")
    groups=[]; labels=[]
    for v,g in d.groupby("var_donor"):
        groups.append(g.margin.to_numpy()); labels.append(str(int(v)))
        value_rows.append({"part":p,"donor_value":int(v),"n":len(g),"median_margin":g.margin.median(),
            "q25_margin":g.margin.quantile(.25),"q75_margin":g.margin.quantile(.75),
            "event_rate":g.responded_but_source_wins.mean()})
    bax.boxplot(groups,tick_labels=labels,showfliers=False,whis=(5,95)); bax.axhline(0,color="black",lw=.8)
    bax.set_xlabel("inserted value"); bax.set_ylabel("final margin"); bax.set_title(f"{p}: value-wise margins")
    for x0,(lab,g) in enumerate(zip(labels,groups),1): bax.text(x0,bax.get_ylim()[1],f"n={len(g)}",ha="center",va="bottom",fontsize=6,rotation=90)
fig.colorbar(im,ax=list(axes[0]),fraction=.015); fig.suptitle("Figure 7 · Exact-value attribution and final-margin distributions")
plt.show(); display(pd.Series(diag,name="diagonal_rate").to_frame().round(3)); display(pd.DataFrame(value_rows).round(3))


### Review record for Figure 7

- **Literal observation:** Post-swap donor-value recognition is graded: diagonal rates are tail 0.272, wing 0.807, beak 0.448, foot 0.905, and eye 0.501. Tail is weakest and foot strongest, rather than all parts failing equally.
- **Strongest alternative explanation:** Different parts have different numbers and frequencies of variants, so raw diagonal rates are not directly interchangeable.
- **Discriminating test:** Relate each donor value to species support and its part's alternative count.
- **Limited conclusion:** `ACCEPTED FOR exact-value difficulty as an additional graded contributor.`
- **Next question:** Do rarity or a larger choice set organize those value-level failures?


## 7b · Are difficult values simply rare or drawn from a larger alternative set?

**Question.** Are difficult values simply rare or drawn from a larger alternative set?

**Variables and prediction.** For every donor value, compare its source-species support with the rate where `response_delta>0` but the final margin remains negative; also report the total number of alternatives for its part. An association supports frequency or choice-set difficulty, but five part-level counts cannot establish a stable correlation.

**Method.** Label every exact value and show its number of swap rows.

### Figure 7b · Are difficult values simply rare or drawn from a larger alternative set?

**How to read the figure.** Each labelled point is one donor value. The x-axis is how many species support
that value; the y-axis is the fraction of its swaps that responded donorward
but still ended source-negative. A consistent downward or upward pattern
would support frequency as an organizer. The number of alternatives is
reported but cannot be cleanly separated with only five parts.


In [ ]:
# ALT: Labelled FunnyBird donor-value plot of species support versus the rate where donor pixels move the margin but the old source remains larger.
VS=(S.groupby(["part","var_donor"]).agg(n_rows=("margin","size"),species_support=("sid_donor","nunique"),
     responded_but_source_wins_rate=("responded_but_source_wins","mean"),median_margin=("margin","median")).reset_index())
VS["alternatives_in_part"]=VS.part.map({p:hi-lo for p,(lo,hi) in SPANS.items()})
fig,ax=plt.subplots(figsize=(9,6))
for p,d in VS.groupby("part"):
    ax.scatter(d.species_support,d.responded_but_source_wins_rate,s=35,color=COLORS[p],label=p)
    for r in d.itertuples(): ax.annotate(f"{p}_{int(r.var_donor)}",(r.species_support,r.responded_but_source_wins_rate),fontsize=6,xytext=(3,3),textcoords="offset points")
ax.set_xlabel("source species carrying donor value"); ax.set_ylabel("fraction: donorward response, but source still wins")
ax.set_ylim(-.02,1.02); ax.legend(); ax.set_title("Figure 7b · Exact-value support versus controlled backwash events")
plt.tight_layout(); plt.show(); display(VS.round(3))


### Review record for Figure 7b

- **Literal observation:** Within each part, source-species support does not show a consistent monotone relationship with the event rate. Tail variants remain high and wing/foot variants low across overlapping support values.
- **Strongest alternative explanation:** The number of alternatives is constant within a part and therefore remains confounded with all other part-level differences.
- **Discriminating test:** Use more independent part families or a design that changes choice-set size while holding pixels and species fixed.
- **Limited conclusion:** `VALID TEST, NO CLEAR SUPPORT that frequency or alternative count alone explains the part ordering.`
- **Next question:** Does unchanged source species organize what remains after exact values?


## 8 · Does source species organize the remaining error after exact values?

**Question.** Does source species organize the remaining error after exact values?

**Variables and prediction.** Subtract the mean margin for each `(part, source value, donor value)` combination, then summarize the residual by source species. Persistent species differences support an additional unchanged-body/species association, but remain observational.

**Method.** Show every part and require at least five rows per displayed species estimate.

### Figure 8 · Does source species organize the remaining error after exact values?

**How to read the figure.** First remove the average margin for the same part, source value, and donor
value. Each remaining point is a source-species mean residual. Zero means that
species behaves like the matched-value average; positive or negative values
mean it systematically shifts the final margin. This is association with the
unchanged bird context, not an independent species manipulation.


In [ ]:
# ALT: Per-source-species FunnyBird margin residuals after controlling exact source and donor values, shown for all parts.
R=S.copy(); R["value_pair_mean"]=R.groupby(["part","var_src","var_donor"]).margin.transform("mean")
R["margin_after_value_pair"]=R.margin-R.value_pair_mean
SP=(R.groupby(["part","sid_src"]).agg(n=("margin","size"),residual=("margin_after_value_pair","mean"))
      .reset_index().query("n>=5"))
fig,axes=plt.subplots(1,5,figsize=(18,4),sharey=True)
for ax,p in zip(axes,ORDER):
    d=SP[SP.part==p].sort_values("residual")
    ax.scatter(np.arange(len(d)),d.residual,color=COLORS[p],s=18)
    ax.axhline(0,color="black",lw=.8); ax.set_title(f"{p} (n species={len(d)})")
    ax.set_xlabel("source species, sorted")
axes[0].set_ylabel("mean margin residual after exact value pair")
fig.suptitle("Figure 8 · Source-species residual after exact source/donor values")
plt.tight_layout(); plt.show(); display(SP.groupby("part").residual.agg(["min","median","max","std","count"]).round(3))


### Review record for Figure 8

- **Literal observation:** After centering each part/source-value/donor-value combination, mean residuals still span roughly 20-35 raw-z units across the 50 source species in every part.
- **Strongest alternative explanation:** The descriptive species means can also absorb pose or repeated-row composition, and they are not a causal body manipulation.
- **Discriminating test:** Check whether species is recoverable from held-out concept vectors and whether species improves held-out margin prediction.
- **Limited conclusion:** `ACCEPTED FOR an observational source-species association beyond exact values.`
- **Next question:** Is species information actually present in the learned concept representation?


## 8b · How much species identity is recoverable from the learned concept vector?

**Question.** How much species identity is recoverable from the learned concept vector?

**Variables and prediction.** Train two small diagnostic species classifiers after the CBM is finished. The grey classifier receives the known binary concept labels c for an image; the colored classifier receives the CBM's learned raw scores z for the same concepts. A bar height is the fraction of held-out images whose species this diagnostic classifier guesses correctly. Above-chance accuracy means species is recoverable from those numbers; it does not say which pixels produced them and is not a grounding score.

**Method.** Use one fixed stratified 70/30 split of the held-out prediction population.

### Figure 8b · How much species identity is recoverable from the learned concept vector?

**How to read the figure.** The y-axis is held-out species-classification accuracy. For every block, one
bar uses the learned raw logits and one uses only the processed 0/1 concept
labels. The label bar is the structural control: with balanced FunnyBird
species and `K` mutually exclusive values for one part, it is approximately
`K/50` (tail has 9 values, so 9/50=0.18), not 1/50. The dashed 1/50 line is
blind guessing; the dotted line is the saved CBM's own species-task accuracy.
Raw-z accuracy above the label-only control is extra within-bucket species
information, but still does not prove that it caused backwash.


### Before Figure 8b: what exactly are the grey and colored bars?

Each image has a processed binary label vector `c`. For example, a row can
contain `beak_0=1`, `beak_1=0`, ..., `wing_3=1`. These are the dataset's
known yes/no concept answers after preprocessing; they are not model scores.

We train a separate diagnostic classifier whose target is the species `y`:

- **grey bar:** input is the corresponding block of known 0/1 labels `c`;
- **colored bar:** input is the corresponding block of learned raw scores `z`;
- **bar height:** held-out species accuracy of that diagnostic classifier.

Thus “species information is present” means only that a classifier can guess
species from the supplied numbers better than chance. It does **not** mean the
saved CBM classified the image with that accuracy, and it does **not** measure
whether a concept used its named pixels.

Example: wing values may be characteristic of particular species, so wing
labels and wing `z` can reveal species. The controlled swap is still required
to ask whether wing `z` follows newly inserted wing pixels.

> **IMPORTANT: Species leakage makes backwash possible, but leakage alone does
> not cause it. Wing is the clearest counterexample: wing `z` reveals species,
> yet the controlled swaps show strong grounding.**

What predicts grounding is measured separately: `response_delta`, final margin
`m_cf`, target-part visibility, label/mask conflict, and exact donor-value
recognition. Figure 8b is an availability/control diagnostic, not that outcome.


In [ ]:
# ALT: Held-out FunnyBird species-decoding accuracy from learned raw concept logits versus known binary concept labels, using matched diagnostic classifiers and the same split.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
y_saved=np.asarray(saved["y"]).reshape(-1).astype(int)
idx=np.arange(len(y_saved)); tr,te=train_test_split(idx,test_size=.30,random_state=20260803,stratify=y_saved)
blocks={"complete raw logits":np.arange(z_saved.shape[1])}
blocks.update({p:np.arange(lo,hi) for p,(lo,hi) in SPANS.items()})
probe=[]
for name,cols in blocks.items():
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(z_saved[tr][:,cols],y_saved[tr]); label_model.fit(c_saved[tr][:,cols],y_saved[tr])
    probe.append({"block":name,
                  "raw_z_accuracy":accuracy_score(y_saved[te],raw_model.predict(z_saved[te][:,cols])),
                  "processed_label_accuracy":accuracy_score(y_saved[te],label_model.predict(c_saved[te][:,cols])),
                  "label_patterns":len(np.unique(c_saved[tr][:,cols],axis=0)),"dimensions":len(cols)})
PROBE=pd.DataFrame(probe)
x=np.arange(len(PROBE)); w=.36; fig,ax=plt.subplots(figsize=(10,5))
ax.bar(x-w/2,PROBE.processed_label_accuracy,w,label="known 0/1 label probe",color="#BBBBBB")
ax.bar(x+w/2,PROBE.raw_z_accuracy,w,label="learned raw-z probe",color=["#333333"]+[COLORS.get(x,"#999999") for x in PROBE.block.iloc[1:]])
ax.set_xticks(x); ax.set_xticklabels(PROBE.block,rotation=25,ha="right")
ax.axhline(1/len(np.unique(y_saved)),color="black",ls="--",label="chance = 1/50")
ax.axhline(task_accuracy,color="#D55E00",ls=":",label=f"saved CBM task accuracy = {task_accuracy:.3f}")
ax.set_ylim(0,1); ax.set_ylabel("held-out species accuracy"); ax.set_title("Figure 8b · Species decoded from learned concept representations")
ax.legend(); plt.tight_layout(); plt.show(); display(PROBE.round(3))


### Review record for Figure 8b

- **Literal observation:** The grey and colored bars now use the same held-out species classifier and split. Grey uses the known binary concept labels c; colored uses the learned raw scores z.
- **Strongest alternative explanation:** Species decodability is not grounding: a score block can identify species while still responding correctly to its named pixels, as the controlled wing swaps show.
- **Discriminating test:** Judge grounding from response_delta and the final donor-minus-source margin, then relate those outcomes to visibility, conflict, and exact-value recognition.
- **Limited conclusion:** `ACCEPTED FOR a paired label-versus-raw-z species-information diagnostic; not a grounding test and not evidence that species information alone causes backwash.`
- **Next question:** How much of the swap margin generalizes from the proposed explanatory blocks?


## 9 · How much does each observed block account for?

**Question.** How much does each observed block account for?

**Variables and prediction.** Predict the raw final margin on held-out render IDs using progressively richer categorical blocks. Lower held-out error means the added block organizes the outcome; remaining error is the measured residual.

**Method.** Use stable five-fold image-level splits, training-fold group means with shrinkage, and no RLv2 variables.

### Figure 9 · How much does each observed block account for?

**How to read the figure.** Panel A compares the median final raw-logit margin for all rows, rows with a
nonzero inserted-part mask, and rows with at least 100 inserted-part pixels;
markers are unconnected because these are nested descriptive selections, not
a trajectory. Panel B is held-out RMSE when predicting final margin; lower is
better. Starting from part alone, blocks are added in order: visibility,
exact source/donor values, then source species. A decrease means the new block
predicts unseen render IDs better. The final nonzero error is the residual;
an increase is negative evidence for that proposed organizer. RMSE 10 to 8
is improvement; RMSE 10 to 11 is not.


In [ ]:
# ALT: Held-out final-margin prediction error after adding FunnyBird visibility, exact values, and source species sequentially.
import hashlib
A=S.copy(); A["vis_bin"]=pd.cut(A.pixel_count_cf,[-1,19,49,99,199,499,np.inf],labels=False)
unit=(A["render_id"].astype(str) if "render_id" in A else
      A.get("li",pd.Series(np.arange(len(A)),index=A.index)).astype(str))
A["fold"]=unit.map(lambda x:int(hashlib.sha1(x.encode()).hexdigest(),16)%5)
stages=[("part only",["part"]),("+ visibility",["part","vis_bin"]),
        ("+ exact values",["part","vis_bin","var_src","var_donor"]),
        ("+ source species",["part","vis_bin","var_src","var_donor","sid_src"])]
rows=[]
for stage,cols in stages:
    pred=pd.Series(index=A.index,dtype=float)
    for fold in range(5):
        tr=A[A.fold!=fold]; te=A[A.fold==fold]
        prior=tr.margin.mean(); stats=tr.groupby(cols).margin.agg(["mean","count"]).reset_index()
        stats["estimate"]=(stats["mean"]*stats["count"]+prior*10)/(stats["count"]+10)
        joined=te[cols].merge(stats[cols+["estimate"]],on=cols,how="left")
        pred.loc[te.index]=joined.estimate.fillna(prior).to_numpy()
    rows.append({"stage":stage,"rmse":float(np.sqrt(np.mean((A.margin-pred)**2))),
                 "mae":float(np.mean(np.abs(A.margin-pred)))})
ACCOUNT=pd.DataFrame(rows)
desc=[]
for p in ORDER:
    d=A[A.part==p]
    for label,mask in [("all rows",np.ones(len(d),dtype=bool)),("visible >0 px",d.pixel_count_cf>0),("clearly visible ≥100 px",d.pixel_count_cf>=100)]:
        g=d[mask]; desc.append({"part":p,"selection":label,"n":len(g),"median_margin":g.margin.median()})
DESC=pd.DataFrame(desc)
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for label,marker,offset in [("all rows","o",-.15),("visible >0 px","s",0),("clearly visible ≥100 px","^",.15)]:
    d=DESC[DESC.selection==label].set_index("part").reindex(ORDER)
    axes[0].scatter(np.arange(len(ORDER))+offset,d.median_margin,label=label,marker=marker,s=45)
axes[0].axhline(0,color="black",lw=.8); axes[0].set_xticks(np.arange(len(ORDER)),ORDER)
axes[0].set_ylabel("median final margin"); axes[0].set_title("A · Descriptive visibility selections"); axes[0].legend(fontsize=8)
axes[1].plot(ACCOUNT.stage,ACCOUNT.rmse,"o-",color="#0072B2")
axes[1].set_ylabel("held-out RMSE of final margin"); axes[1].tick_params(axis="x",rotation=25)
axes[1].set_title("B · Sequential held-out accounting")
fig.suptitle("Figure 9 · What measured contributors organize, and what remains")
plt.tight_layout(); plt.show(); display(DESC.round(3)); display(ACCOUNT.round(3))


### Review record for Figure 9

- **Literal observation:** Held-out RMSE improves only from 11.467 to 11.106 when visibility is added. It then worsens to 11.225 with exact values and 12.062 with source species.
- **Strongest alternative explanation:** High-cardinality categorical means may be too sparse or poorly regularized, but that possibility cannot be counted as positive evidence.
- **Discriminating test:** Use seed replication or a preregistered hierarchical/regularized predictor before assigning generalizing explanatory credit to exact values or species.
- **Limited conclusion:** `VALID TEST, NO SUPPORT from this predictor that exact values or source species account for held-out margin variance; only visibility gives a small improvement.`
- **Next question:** Does the concept-layer margin have a large downstream class consequence?


## 9b · Do the proposed contributors line up with the controlled part ordering?

**Question.** Do the proposed contributors line up with the controlled part ordering?

**Variables and prediction.** Place four separately defined part-level quantities in aligned panels: the controlled backwash-candidate rate, the same rate among swaps with at least 100 target pixels, the training label/mask conflict rate, and one minus exact donor-value recognition. Tail should be high across several contributor panels while wing and foot should be low if the proposed explanation matches the controlled outcome. The panels use different units and must not be added together.

**Method.** Use the same five-part order in every panel and print the exact table.

### Figure 9b · Do the proposed contributors line up with the controlled part ordering?

**How to read the figure.** All four panels use the same y-axis part order. Panel A is the fraction of
all swaps satisfying `response_delta>0 and m_cf<0`. Panel B repeats that
fraction only when the inserted target occupies at least 100 pixels. Panel C
is the fraction of original positive training labels removed by the matched
visibility rule. Panel D is one minus the post-swap inserted-value recognition
rate. Larger is worse in every panel, but the denominators and meanings differ,
so the bar heights must not be added. The shared ordering asks whether the
proposed contributors align with the controlled outcome.


In [ ]:
# ALT: Four aligned FunnyBird part-level panels comparing the controlled backwash outcome with clear-visibility residuals, label/mask conflict, and exact donor-value error.
if "PART_CONFLICT" not in globals():
    raise RuntimeError("Figure 9b requires the matched standard/RLv2 label records used in Figure 6b")
if "diag" not in globals():
    raise RuntimeError("Figure 9b requires the exact-value recognition results from Figure 7")
FB_SYN=pd.DataFrame(index=ORDER)
FB_SYN.index.name="part"
FB_SYN["controlled_backwash_rate"]=(S.groupby("part").responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["clear_visible_backwash_rate"]=(S[S.pixel_count_cf>=100].groupby("part")
                                          .responded_but_source_wins.mean().reindex(ORDER))
FB_SYN["label_mask_conflict_rate"]=PART_CONFLICT.conflict_rate.reindex(ORDER)
FB_SYN["donor_value_error_rate"]=pd.Series({p:1-diag[p] for p in ORDER}).reindex(ORDER)
panels=[
    ("controlled_backwash_rate","A · controlled outcome"),
    ("clear_visible_backwash_rate","B · outcome when target ≥100 px"),
    ("label_mask_conflict_rate","C · training label/mask conflict"),
    ("donor_value_error_rate","D · inserted value not recognized"),
]
fig,axes=plt.subplots(1,4,figsize=(16,4.5),sharey=True)
for ax,(column,title) in zip(axes,panels):
    ax.barh(np.arange(len(ORDER)),FB_SYN[column],color=[COLORS[p] for p in ORDER])
    ax.set_xlim(0,1); ax.set_title(title,fontsize=10); ax.set_xlabel("fraction")
    ax.set_yticks(np.arange(len(ORDER)),ORDER); ax.invert_yaxis()
fig.suptitle("Figure 9b · Controlled FunnyBird outcome and proposed contributors in one part order")
plt.tight_layout(); plt.show(); display(FB_SYN.round(3))
from IPython.display import Markdown
display(Markdown(
    "**Literal observation.** Tail has the largest controlled backwash rate, the "
    "largest training conflict rate, and the largest exact-value error. Beak and "
    "eye are intermediate in the controlled outcome and value error; wing and foot "
    "are lowest in the controlled outcome despite species being decodable from their "
    "raw scores. Clear visibility reduces but does not erase every failure.\n\n"
    "**Limited conclusion.** The contributors align with the controlled part ordering, "
    "but these fractions are not additive and this figure does not claim that their sum "
    "explains every swap row."
))


## 10 · Does the concept-layer error materially alter species prediction?

**Question.** Does the concept-layer error materially alter species prediction?

**Variables and prediction.** Relate final concept margin to the model's donor-species probability, which is a different downstream quantity. A small downstream change would limit the harm to explanation reliability rather than widespread class failure.

**Method.** Use independent final-margin bins and print bin counts.

### Figure 10 · Does the concept-layer error materially alter species prediction?

**How to read the figure.** Swaps are divided into ten non-overlapping, approximately equal-count bins by final donor-minus-source concept
margin on the x-axis. The y-axis is the model's mean probability for the donor
species, with the number of rows printed per bin. This asks whether concept
grounding failure has a downstream class effect; it is intentionally the one
place where class probability, rather than raw concept `z`, is the outcome.


In [ ]:
# ALT: Binned relationship between FunnyBird final concept margin and downstream donor-species probability.
prob_col=next((c for c in ["p_cf_donor","p_donor_cf","donor_species_prob"] if c in S),None)
if prob_col is None:
    print("INCOMPLETE: swap CSV has no donor-species probability column")
else:
    D=S.copy(); D["margin_bin"]=pd.qcut(D.margin,10,duplicates="drop")
    Q=D.groupby("margin_bin",observed=True).agg(n=(prob_col,"size"),mean_margin=("margin","mean"),mean_donor_species_prob=(prob_col,"mean")).reset_index()
    fig,ax=plt.subplots(figsize=(7,4)); ax.plot(Q.mean_margin,Q.mean_donor_species_prob,"o-")
    for r in Q.itertuples(): ax.annotate(f"n={r.n}",(r.mean_margin,r.mean_donor_species_prob),fontsize=7)
    ax.axvline(0,color="black",lw=.8); ax.set_xlabel("mean final concept margin in bin")
    ax.set_ylabel("mean donor-species probability"); ax.set_title("Figure 10 · Downstream consequence of the concept margin")
    plt.tight_layout(); plt.show(); display(Q.round(3))


### Review record for Figure 10

- **Literal observation:** Mean donor-species probability rises from approximately zero at negative margins to 0.107 in the most donor-positive bin, with about 500 rows per bin.
- **Strongest alternative explanation:** A one-part replacement need not make the whole donor species plausible because the unchanged body and other parts still belong to the source.
- **Discriminating test:** Replicate across seeds and compare class-logit changes, not only final probability.
- **Limited conclusion:** `ACCEPTED FOR a monotone but modest single-swap downstream donor-species effect; the primary harm here is explanation fidelity.`
- **Next question:** Does minimality change the accepted standard-CBM quantities?


## 11 · Standard-CBM evidence ledger

Figures 1–10 were displayed and reviewed together on 2026-08-04.

| Predicate or explanation | Direct measurement | Status after review |
|---|---|---|
| model outputs are usable | Figure 1 | `ACCEPTED FOR seed-1 model health` |
| interventions are valid | Figure 2 | `ACCEPTED FOR validated fixed renders` |
| inserted pixels cause donorward movement | Figure 3 | `ACCEPTED FOR all five parts` |
| old source can remain stronger after that movement | Figure 4 | `ACCEPTED; strongest for tail, beak, eye` |
| direction artifact excluded | Figure 5 | `ACCEPTED` |
| visibility contribution | Figure 6 | `ACCEPTED AS CONTRIBUTOR, NOT SUFFICIENT` |
| training label/mask conflict measured | Figure 6b | `MEASURED; causal effect deferred to RLv2` |
| exact-value contribution | Figure 7 | `ACCEPTED AS GRADED CONTRIBUTOR` |
| frequency/alternative-count explanation | Figure 7b | `VALID TEST, NO CLEAR SUPPORT AS SOLE EXPLANATION` |
| source-species residual | Figure 8 | `OBSERVATIONAL ASSOCIATION` |
| species information beyond concept-label buckets | Figure 8b | `INCOMPLETE: PAIRED RAW-Z/LABEL CONTROL REQUIRES REVIEW` |
| sequential descriptive accounting | Figure 9 | `VISIBILITY IMPROVES; EXACT VALUES/SPECIES DO NOT` |
| downstream class consequence | Figure 10 | `MONOTONE BUT MODEST DONOR-PROBABILITY EFFECT` |

**Next report question.** Notebook 03 asks whether the MCBM minimality
penalty changes these same accepted quantities. It must not replace the
standard-CBM result established here.


# Methods appendix · measurements not used in the main claim

The reciprocal mask-deletion and randomized-patch experiments are retained
as method-development history. They did not reproduce the clean FunnyBird
control sufficiently to transfer their causal interpretation to CUB.

- reciprocal mask deletion: `METHOD NOT CALIBRATED FOR CROSS-DATASET CAUSAL COMPARISON`;
- randomized patch V1/V2: local pixel response was measurable in selected
  examples, but the all-part control was not calibrated and wing coverage was
  inadequate;
- none of these outcomes invalidates the validated renderer swap above.

Full artifacts and scripts remain under `analysis/paired_mask_deletion.py`,
`analysis/randomized_patch_masking.py`, and their output directories. They
are not rerun by this notebook.


# Provenance appendix

The table below records the live Git commit, input paths and SHA-256
hashes, row counts, seed, and the accepted fixed-render root. It is part
of the report: a stale HTML is not synchronized evidence.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
commit=subprocess.run(["git","rev-parse","HEAD"],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
prov=[]
for role,path in [("fixed-render swap CSV",SWAP),("prediction export",PRED),("model checkpoint",MODEL)]:
    prov.append({"role":role,"path":str(path),"sha256":sha256_file(path)})
display(pd.DataFrame(prov)); display(pd.DataFrame([{"git_commit":commit,"seed":1,
    "swap_rows":len(S),"prediction_images":len(c_saved),"exact_concepts":len(CONCEPT_NAMES),
    "excluded_swap_rows":0,"accepted_render_root":str(SWAP.parent)}]))
